# Apply, Map, and Pipe

Pandas provides several methods for applying custom transformations to data. Understanding when to use each—and when to avoid them—is crucial for writing efficient and readable code. This notebook covers `.map()`, `.apply()`, and `.pipe()` with performance considerations.

## Learning Objectives

- Use `.map()` for element-wise transformations on Series
- Understand `.apply()` on Series vs DataFrame (axis parameter)
- Know when `.apply()` is slow and what to use instead
- Master `.pipe()` for readable method chaining
- Write reusable transformation functions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
# Load data
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

## .map() on Series

The `.map()` method applies a function or dictionary mapping to each element in a Series.

In [ ]:
# Map with a dictionary - value replacement
class_names = {1: 'First', 2: 'Second', 3: 'Third'}
df['Pclass_name'] = df['Pclass'].map(class_names)

print("Mapped class names:")
df[['Pclass', 'Pclass_name']].head(10)

In [ ]:
# Map with a function
df['Sex_initial'] = df['Sex'].map(lambda x: x[0].upper())

print("First letter of sex:")
df[['Sex', 'Sex_initial']].head()

In [ ]:
# Map with a Series (for lookups)
port_cities = pd.Series({
    'S': 'Southampton',
    'C': 'Cherbourg',
    'Q': 'Queenstown'
})

df['Port_name'] = df['Embarked'].map(port_cities)
print("Mapped port names:")
df[['Embarked', 'Port_name']].head(10)

> **Gotcha: map() vs replace()**
> 
> - `.map()` returns NaN for values not in the mapping dictionary
> - `.replace()` keeps original values for non-matches
> 
> Use `.replace()` when you want to change only some values.

In [ ]:
# Difference between map and replace
partial_map = {1: 'First'}  # Only maps class 1

print("map() with partial mapping (others become NaN):")
print(df['Pclass'].map(partial_map).head(10))

print("\nreplace() with partial mapping (others unchanged):")
print(df['Pclass'].replace(partial_map).head(10))

## .apply() on Series

In [ ]:
# Apply a function to each element
def categorize_age(age):
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['Age_category'] = df['Age'].apply(categorize_age)
print("Age categories:")
print(df['Age_category'].value_counts())

## .apply() on DataFrame

When applied to a DataFrame, `.apply()` works on rows or columns depending on the `axis` parameter.

In [ ]:
# Apply to columns (axis=0, default)
numeric_cols = df[['Age', 'Fare', 'SibSp', 'Parch']]

# Range of each column
print("Range of each column (axis=0):")
print(numeric_cols.apply(lambda col: col.max() - col.min()))

In [ ]:
# Apply to rows (axis=1)
df['Family_size'] = df.apply(lambda row: row['SibSp'] + row['Parch'] + 1, axis=1)

print("Family size calculated with apply:")
df[['SibSp', 'Parch', 'Family_size']].head()

> **Gotcha: apply() with axis=1 is SLOW**
> 
> Applying a function row-by-row is essentially a Python for-loop. For most operations, there's a faster vectorized alternative.

In [ ]:
# Better: vectorized version (much faster!)
df['Family_size_vec'] = df['SibSp'] + df['Parch'] + 1

# Verify they're the same
print("Results are identical:", (df['Family_size'] == df['Family_size_vec']).all())

## Performance: apply() vs Vectorized Operations

In [ ]:
# Create a large DataFrame for benchmarking
n_rows = 100000
large_df = pd.DataFrame({
    'A': np.random.randn(n_rows),
    'B': np.random.randn(n_rows),
    'C': np.random.randn(n_rows)
})

print(f"Benchmarking with {n_rows:,} rows...")

In [ ]:
# Benchmark: calculate row sum three ways
import time

# Method 1: apply()
start = time.perf_counter()
result1 = large_df.apply(lambda row: row['A'] + row['B'] + row['C'], axis=1)
apply_time = time.perf_counter() - start

# Method 2: vectorized
start = time.perf_counter()
result2 = large_df['A'] + large_df['B'] + large_df['C']
vec_time = time.perf_counter() - start

# Method 3: .sum(axis=1)
start = time.perf_counter()
result3 = large_df.sum(axis=1)
sum_time = time.perf_counter() - start

print(f"apply(axis=1):  {apply_time:.4f}s")
print(f"vectorized:     {vec_time:.4f}s")
print(f".sum(axis=1):   {sum_time:.4f}s")
print(f"\nSpeedup: {apply_time/vec_time:.0f}x faster with vectorized operations!")

In [ ]:
# Visualize the performance difference
methods = ['apply(axis=1)', 'vectorized', '.sum(axis=1)']
times = [apply_time, vec_time, sum_time]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods, times, color=['#e74c3c', '#2ecc71', '#3498db'])
ax.set_ylabel('Time (seconds)')
ax.set_title(f'Performance Comparison: Row Sum on {n_rows:,} Rows')

# Add value labels
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{t:.4f}s', ha='center', va='bottom')

# Log scale to see the difference
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## Alternatives to apply()

Most common `apply()` operations have vectorized alternatives.

In [ ]:
# Instead of: df.apply(lambda row: 'Yes' if row['Survived'] == 1 else 'No', axis=1)
# Use np.where:
df['Survived_text'] = np.where(df['Survived'] == 1, 'Yes', 'No')

print("Using np.where:")
df[['Survived', 'Survived_text']].head()

In [ ]:
# For multiple conditions, use np.select:
conditions = [
    df['Age'] < 18,
    df['Age'] < 60,
    df['Age'] >= 60
]
choices = ['Child', 'Adult', 'Senior']

df['Age_cat_vec'] = np.select(conditions, choices, default='Unknown')

print("Using np.select:")
print(df['Age_cat_vec'].value_counts())

## .pipe() for Method Chaining

The `.pipe()` method enables readable method chaining by passing the DataFrame as the first argument to a function.

In [ ]:
# Without pipe - hard to read
def add_family_size(df):
    df = df.copy()
    df['family_size'] = df['SibSp'] + df['Parch'] + 1
    return df

def add_is_alone(df):
    df = df.copy()
    df['is_alone'] = df['family_size'] == 1
    return df

def add_fare_per_person(df):
    df = df.copy()
    df['fare_per_person'] = df['Fare'] / df['family_size']
    return df

# Nested function calls - ugly!
result_ugly = add_fare_per_person(add_is_alone(add_family_size(df)))

In [ ]:
# With pipe - clean and readable!
result_clean = (
    df
    .pipe(add_family_size)
    .pipe(add_is_alone)
    .pipe(add_fare_per_person)
)

print("Pipeline result:")
result_clean[['Name', 'SibSp', 'Parch', 'family_size', 'is_alone', 'Fare', 'fare_per_person']].head()

In [ ]:
# Complete data cleaning pipeline
def load_and_clean(filepath):
    return pd.read_csv(filepath)

def select_columns(df, cols):
    return df[cols]

def fill_missing_age(df, strategy='median'):
    df = df.copy()
    if strategy == 'median':
        df['Age'] = df['Age'].fillna(df['Age'].median())
    return df

def add_features(df):
    df = df.copy()
    df['family_size'] = df['SibSp'] + df['Parch'] + 1
    df['is_child'] = df['Age'] < 18
    return df

def drop_na(df, subset=None):
    return df.dropna(subset=subset)

# The full pipeline
clean_df = (
    pd.read_csv('../../data/titanic.csv')
    .pipe(select_columns, ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare'])
    .pipe(fill_missing_age, strategy='median')
    .pipe(add_features)
    .pipe(drop_na)
)

print(f"Clean data shape: {clean_df.shape}")
clean_df.head()

## Writing Reusable Transformation Functions

In [ ]:
# Good practice: functions that work with pipe()
def log_shape(df, message=""):
    """Log DataFrame shape for debugging pipelines."""
    print(f"{message}Shape: {df.shape}")
    return df

def filter_by_column(df, column, values):
    """Filter DataFrame to rows where column is in values."""
    return df[df[column].isin(values)]

def rename_columns_lower(df):
    """Rename all columns to lowercase."""
    return df.rename(columns=str.lower)

# Use in pipeline
result = (
    df
    .pipe(log_shape, "Original: ")
    .pipe(filter_by_column, 'Pclass', [1, 2])
    .pipe(log_shape, "After filter: ")
    .pipe(rename_columns_lower)
)

print("\nFinal columns:", list(result.columns))

## Exercises

### Exercise 1: Map Practice

Create a dictionary mapping and use `.map()` to add a 'Title' column extracted from the Name column (Mr, Mrs, Miss, Master, etc.).

In [ ]:
# Extract and map titles
# Hint: Use .str.extract() to get titles first
# YOUR CODE HERE

### Exercise 2: Replace apply() with Vectorized

The following code uses apply(). Rewrite it using vectorized operations.

In [ ]:
# Original slow code
def categorize_fare(row):
    if row['Fare'] < 10:
        return 'Budget'
    elif row['Fare'] < 50:
        return 'Standard'
    else:
        return 'Premium'

# Rewrite using np.select or pd.cut
# YOUR CODE HERE

### Exercise 3: Build a Pipeline

Create a data cleaning pipeline with pipe() that:
1. Loads the Titanic data
2. Drops the 'Cabin' column
3. Fills missing Age with median
4. Adds a 'deck' column from Ticket (first letter)
5. Filters to only passengers who paid > 0 fare

In [ ]:
# Build the pipeline
# YOUR CODE HERE

### Exercise 4: Performance Comparison

Create a function that can be called with apply() and also implement a vectorized version. Benchmark both on the large_df created earlier. The function should calculate: (A^2 + B^2)^0.5 (Euclidean distance from origin).

In [ ]:
# Implement both versions and benchmark
# YOUR CODE HERE